# ML-07 — Baseline Action Score and Top-20 Review

Simple words, honest numbers. Everything below is **observed / measured / directional / decision-support**, not a claim about Google's algorithm.

**Data:** the 30k-row anonymized starter slice (`data/raw/content_refresh_anonymized.csv`). No client names, URLs or queries are printed here.

**Outcome used to check signals and score the queue:** `is_declining_label = (trend_direction == "down")`, meaning impressions in the last 30 days fell more than 20% versus the 30 days before. The label is used **only to evaluate**. It is never an input to the rule.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# find the repo root whether we run locally or in Colab
here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p / "data" / "raw").exists()), None)
assert ROOT is not None, "Run this from inside your repo (data/raw/ not found)."

DATA = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
OUT = ROOT / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA)
df["declining"] = (df["trend_direction"] == "down").astype(int)   # evaluation label ONLY
BASE_RATE = df["declining"].mean()
print(df.shape, "| base rate of declining pages:", round(BASE_RATE, 3))

(30000, 45) | base rate of declining pages: 0.542


## 1. My rule and its reason codes

### 1a. Check two signals first

I want to build a rule around **CTR that is low for the page's position**. Before I trust it, I test it, and I test one more signal from the session (**staleness**). Each check is one bucket table with `n` printed, and a one word verdict.

Reminders from the data dictionary: `ctr` is a ×100 percentage (0.5 means 0.5%), and `avg_position = 0` means "no position data", so those rows are dropped for the CTR check.

In [2]:
# ---- Signal 1: low CTR for its position (behind FlyRank's CTR-vs-position flag) ----
visible = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)].copy()
visible["position_band"] = pd.cut(visible["avg_position"], [0, 3, 10, 20], labels=["1-3", "4-10", "11-20"])
visible["ctr_bucket"] = np.where(visible["ctr"] < 0.5, "low (<0.5%)", "ok (>=0.5%)")

sig1 = (visible.groupby(["position_band", "ctr_bucket"], observed=True)["declining"]
        .agg(n="size", decline_rate="mean").round(3).reset_index())
print("Signal 1: decline rate by CTR bucket, inside each position band (visible pages only)")
print(sig1.to_string(index=False))

overall = visible.groupby("ctr_bucket")["declining"].agg(n="size", decline_rate="mean").round(3)
print("\nPooled:"); print(overall)

Signal 1: decline rate by CTR bucket, inside each position band (visible pages only)
position_band  ctr_bucket    n  decline_rate
          1-3 low (<0.5%)  360         0.856
          1-3 ok (>=0.5%)  120         0.383
         4-10 low (<0.5%) 5609         0.611
         4-10 ok (>=0.5%) 1475         0.458
        11-20 low (<0.5%) 3790         0.630
        11-20 ok (>=0.5%)  669         0.531

Pooled:
                n  decline_rate
ctr_bucket                     
low (<0.5%)  9759         0.627
ok (>=0.5%)  2264         0.475


**Signal 1 verdict: CONFIRMED (directional).** Inside every position band, the low-CTR pages decline more often than the ok-CTR pages, so this is not just a position effect. The gap is biggest at positions 1 to 3 (about 86% vs 38%, but only 120 pages in the "ok" cell, so I treat the size of that gap with care). It holds at 4 to 10 and 11 to 20 with large n. The rule idea survives this check. Caveat: this is an association on one slice, and clients differ a lot in their decline rates (see section 4).

In [3]:
# ---- Signal 2: staleness (behind FlyRank's stale_visible_page flag) ----
stale = df[df["impressions_90d"] >= 500]
order = ["0-30", "31-90", "91-180", "181+"]
sig2 = (stale.groupby("freshness_tier")["declining"]
        .agg(n="size", decline_rate="mean").round(3).reindex(order).reset_index())
print("Signal 2: decline rate by days since last update (pages with >= 500 impressions)")
print(sig2.to_string(index=False))

Signal 2: decline rate by days since last update (pages with >= 500 impressions)
freshness_tier     n  decline_rate
          0-30 10063         0.583
         31-90    88         0.523
        91-180  6558         0.616
          181+    17         0.941


**Signal 2 verdict: MIXED.** Pages last updated 91 to 180 days ago decline a bit more (about 62%) than pages updated in the last 30 days (about 58%), which points the expected way, but the effect is small. The 31 to 90 bucket goes the other way (n is only 88), and the 181+ bucket looks very high but has n = 17, far too few to trust. So staleness is a weak, unclear signal in this slice. Because of this I did **not** build my rule on it. That is a useful negative: it saved me from a rule that would have leaned on 17 pages.

### 1b. The rule in plain words

> A page is worth a title and meta description review first if it already gets a lot of impressions, ranks on page one or two (position 20 or better), and its CTR is lower than what healthy pages at the same position get. The more clicks it is likely leaving on the table, the higher it ranks.

**Score:** `impressions_90d × (expected_ctr − ctr) / 100`, which is roughly "estimated clicks left on the table in 90 days". `expected_ctr` is the 75th percentile CTR of visible pages in the same position band. Only pages with ≥ 500 impressions and position 0 to 20 are scored.

**Reason code (only one):** `LOW_CTR_FOR_POSITION`
**Action label:** `rewrite_title_and_meta`

## 2. Build the ranked queue (writes the CSV)

In [4]:
# inputs the rule is allowed to use (all are 90-day totals / rates, none are label-derived)
RULE_INPUTS = ["impressions_90d", "avg_position", "ctr"]

BANDS = [0, 3, 10, 20]
LABELS = ["1-3", "4-10", "11-20"]

eligible = (df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)
df["position_band"] = pd.cut(df["avg_position"], BANDS, labels=LABELS)

# expected CTR per band = what a healthy page at that position gets (75th percentile)
expected = df[eligible].groupby("position_band", observed=True)["ctr"].quantile(0.75)
print("expected CTR (%) by band:\n", expected.round(3).to_string())

df["expected_ctr"] = df["position_band"].map(expected).astype(float)
df["ctr_gap"] = (df["expected_ctr"] - df["ctr"]).clip(lower=0)

df["score"] = np.where(eligible, df["impressions_90d"] * df["ctr_gap"] / 100, 0.0)
df["reason_code"] = np.where(df["score"] > 0, "LOW_CTR_FOR_POSITION", "NOT_FLAGGED")
df["action"] = np.where(df["score"] > 0, "rewrite_title_and_meta", "no_action")

queue = df[df["score"] > 0].sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

cols = ["rank", "content_id", "client_id", "score", "reason_code", "action",
        "impressions_90d", "clicks_90d", "avg_position", "ctr", "expected_ctr"]
queue[cols].to_csv(OUT / "baseline_action_score.csv", index=False)
print(f"\nflagged {len(queue):,} of {len(df):,} pages -> wrote {OUT / 'baseline_action_score.csv'}")
queue[cols].head(10)

expected CTR (%) by band:
 position_band
1-3      0.492
4-10     0.440
11-20    0.340

flagged 8,953 of 30,000 pages -> wrote /run/media/zunkode/AIML_Studio/flyrank-ml-internship/work/outputs/baseline_action_score.csv


,rank,content_id,client_id,score,reason_code,action,impressions_90d,clicks_90d,avg_position,ctr,expected_ctr
0,1,content_8c19996aa890,client_4e07408562,1744.1881,LOW_CTR_FOR_POSITION,rewrite_title_and_meta,509252,785,2.5,0.15,0.4925
1,2,content_5fe46e04994d,client_4e07408562,1553.1450,LOW_CTR_FOR_POSITION,rewrite_title_and_meta,517715,741,4.2,0.14,0.4400
2,3,content_8451fc6f034d,client_d029fa3a95,1258.6660,LOW_CTR_FOR_POSITION,rewrite_title_and_meta,272144,75,2.3,0.03,0.4925
3,4,content_36ff89c8214e,client_19581e27de,1150.8783,LOW_CTR_FOR_POSITION,rewrite_title_and_meta,295097,154,7.3,0.05,0.4400
4,5,content_aaef01a50def,client_19581e27de,982.5071,LOW_CTR_FOR_POSITION,rewrite_title_and_meta,517109,1270,5.4,0.25,0.4400
5,6,content_c8e9d6ab9013,client_19581e27de,918.1832,LOW_CTR_FOR_POSITION,rewrite_title_and_meta,208678,0,9.7,0.00,0.4400
6,7,content_c84a0ab98e90,client_f369cb89fc,915.4111,LOW_CTR_FOR_POSITION,rewrite_title_and_meta,223271,70,7.8,0.03,0.4400
7,8,content_1a9e894be2e2,client_19581e27de,873.9780,LOW_CTR_FOR_POSITION,rewrite_title_and_meta,416180,944,4.0,0.23,0.4400
8,9,content_cb112fce36be,client_19581e27de,867.7480,LOW_CTR_FOR_POSITION,rewrite_title_and_meta,309910,492,5.6,0.16,0.4400
9,10,content_db5989a78dd3,client_4e07408562,793.7553,LOW_CTR_FOR_POSITION,rewrite_title_and_meta,345111,733,5.4,0.21,0.4400


In [5]:
# precision@K next to the base rate (a number means little without it)
def precision_at_k(k):
    return queue["declining"].head(k).mean()

eligible_rate = df.loc[eligible, "declining"].mean()
rows = [{"K": k, "precision@K": round(precision_at_k(k), 3)} for k in (10, 20, 50, 100, 500)]
print(pd.DataFrame(rows).to_string(index=False))
print("\nbase rate, all pages:          ", round(BASE_RATE, 3))
print("base rate, eligible pages only:", round(eligible_rate, 3))

metrics = {
    "rows": int(len(df)),
    "flagged": int(len(queue)),
    "base_rate_all": round(float(BASE_RATE), 4),
    "base_rate_eligible": round(float(eligible_rate), 4),
    "precision_at_k": {str(k): round(float(precision_at_k(k)), 4) for k in (10, 20, 50, 100, 500)},
    "signal_verdicts": {"low_ctr_for_position": "CONFIRMED", "staleness": "MIXED"},
    "rule_inputs": RULE_INPUTS,
}
(OUT / "baseline_metrics.json").write_text(json.dumps(metrics, indent=2))

  K  precision@K
 10        0.500
 20        0.400
 50        0.540
100        0.450
500        0.474

base rate, all pages:           0.542
base rate, eligible pages only: 0.599


375

**How to read this:** the baseline is honestly weak. Precision at 10, 20 and 50 sits around or below the base rate of the eligible pages (about 0.60), and near the base rate for all pages (0.54). It picks pages with a real CTR gap, but a CTR gap alone does not tell me which pages will keep losing impressions. That is the bar for the Week 5 model to beat, and I am freezing this rule now so I do not move the goalposts later.

## 3. Top-20 review

For each of the top 20: the action, the reason code, a confidence note, and what would make it wrong. The "what would make it wrong" line is chosen from the row's own numbers. I read each one and edit the notes below if I disagree.

In [6]:
def review(row):
    conf = "medium" if row["avg_position"] <= 10 else "lower (position 11-20, more room for noise)"
    wrong = []
    if row["clicks_90d"] == 0:
        wrong.append("0 clicks on a lot of impressions looks more like a tracking or indexing issue than a weak title")
    if row["avg_position"] <= 3:
        wrong.append("at positions 1-3 a low CTR may come from SERP features taking the click, not the title")
    if str(row.get("main_intent")) == "navigational":
        wrong.append("navigational queries often click another site, so low CTR is normal")
    if not wrong:
        wrong.append("seasonal demand or SERP features could lower CTR no matter how the title reads")
    return conf, "; ".join(wrong)

top20 = queue.head(20).copy()
top20[["confidence", "what_would_make_it_wrong"]] = top20.apply(lambda r: pd.Series(review(r)), axis=1)

for _, r in top20.iterrows():
    print(f"#{r['rank']:>2}  {r['action']} | {r['reason_code']} | pos {r['avg_position']}, "
          f"ctr {r['ctr']}%, impressions {int(r['impressions_90d']):,}, clicks {int(r['clicks_90d']):,}")
    print(f"     confidence: {r['confidence']}")
    print(f"     wrong if: {r['what_would_make_it_wrong']}")

# 1  rewrite_title_and_meta | LOW_CTR_FOR_POSITION | pos 2.5, ctr 0.15%, impressions 509,252, clicks 785
     confidence: medium
     wrong if: at positions 1-3 a low CTR may come from SERP features taking the click, not the title
# 2  rewrite_title_and_meta | LOW_CTR_FOR_POSITION | pos 4.2, ctr 0.14%, impressions 517,715, clicks 741
     confidence: medium
     wrong if: seasonal demand or SERP features could lower CTR no matter how the title reads
# 3  rewrite_title_and_meta | LOW_CTR_FOR_POSITION | pos 2.3, ctr 0.03%, impressions 272,144, clicks 75
     confidence: medium
     wrong if: at positions 1-3 a low CTR may come from SERP features taking the click, not the title
# 4  rewrite_title_and_meta | LOW_CTR_FOR_POSITION | pos 7.3, ctr 0.05%, impressions 295,097, clicks 154
     confidence: medium
     wrong if: seasonal demand or SERP features could lower CTR no matter how the title reads
# 5  rewrite_title_and_meta | LOW_CTR_FOR_POSITION | pos 5.4, ctr 0.25%, impressions 517,109,

### My reading of the top 10

*(Edit this after you read the printout above. Keep it to one line per row.)*

- Most of the top 10 are big pages at position 2 to 8 with a CTR far below the healthy level for that position. The action (rewrite title and meta) fits, and it is cheap to try.
- Row 6 has 0 clicks on about 209k impressions, and rows 3, 4 and 7 have under 160 clicks on 220k+ impressions. Those are the ones I trust least. That pattern can mean a data or indexing issue rather than a copy problem, so I would check Search Console before rewriting.
- Rows 1 and 3 sit at position 2 to 3, where SERP features could be taking the clicks, so a new title may not help there.

## 4. Weak picks + leakage check

In [7]:
top50 = queue.head(50)
print("Top 50 by client (share of the top 50 that comes from each client):")
print(top50["client_id"].value_counts(normalize=True).head(4).round(2).to_string())
print("\nDecline rate by client, among eligible pages (largest clients):")
big = df[eligible].groupby("client_id")["declining"].agg(n="size", decline_rate="mean").round(3)
print(big.sort_values("n", ascending=False).head(5).to_string())
print("\nDecline rate by impression tier (all pages):")
print(df.groupby("impression_tier")["declining"].agg(n="size", decline_rate="mean").round(3).to_string())

Top 50 by client (share of the top 50 that comes from each client):
client_id
client_19581e27de    0.56
client_4e07408562    0.16
client_f369cb89fc    0.12
client_349c41201b    0.08

Decline rate by client, among eligible pages (largest clients):
                      n  decline_rate
client_id                            
client_19581e27de  4395         0.549
client_6208ef0f77  1544         0.613
client_4e07408562  1511         0.471
client_3fdba35f04  1060         0.796
client_7f2253d7e2   594         0.960

Decline rate by impression tier (all pages):
                     n  decline_rate
impression_tier                     
excellent         1078         0.462
good              7205         0.586
low              11248         0.454
moderate         10469         0.615


**Weak picks, in plain words:**

1. **One client dominates the queue.** A big share of the top 50 comes from a single client, because that client has many very large pages. The score rewards raw impressions, so it favours big sites. A per-client cap or a client-level rank would be fairer, and I note it for the model stage instead of changing the frozen baseline.
2. **The biggest pages do not decline the most.** The `excellent` impression tier has a lower decline rate than `moderate` or `good`. Because my score scales with impressions, it pushes toward pages that are less likely to decline.
3. **Clients differ a lot in decline rate.** Among eligible pages the big clients range from about 0.47 to about 0.96. Some of what the rule "finds" may just be which client a page belongs to.
4. **Zero click rows.** Row 6 in the top 20 has 0 clicks on about 209k impressions. I would verify rows like that before acting.

In [8]:
# ---- leakage check ----
FORBIDDEN = {"trend_direction", "trend_pct", "is_declining_label", "declining",
             "impressions_last_30d", "impressions_prev_30d", "clicks_last_30d", "clicks_prev_30d",
             "sessions_last_30d", "sessions_prev_30d", "content_id", "client_id"}

leaked = FORBIDDEN & set(RULE_INPUTS)
assert not leaked, f"leakage: {leaked}"
print("Rule inputs:", RULE_INPUTS)
print("No label-derived columns, no 30-day windows, no IDs used as inputs. OK")

Rule inputs: ['impressions_90d', 'avg_position', 'ctr']
No label-derived columns, no 30-day windows, no IDs used as inputs. OK


**Leakage check, honestly:** the rule uses only `impressions_90d`, `avg_position` and `ctr`. It does **not** use `trend_direction`, `trend_pct`, any `*_last_30d` or `*_prev_30d` column, or any ID. The label is used only to score the queue afterwards. One caveat I cannot remove in this slice: the 90-day totals include the last 30 days, which is the period the label is measured on. So the totals overlap the label window a little. That is why I call every result here **directional**, and why the Week 5 model should be tested with a proper forward window on the full release.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.